# PyTorch GPU vs. CPU Deep Learning Stress Test & Hardware Telemetry Benchmark
A self-contained benchmark notebook that trains a **Deep Residual Neural Network** on both the **CPU** and an **NVIDIA CUDA GPU** while sampling real-time hardware telemetry (`nvidia-smi` + PyTorch memory counters + system CPU threads).

### Benchmark Pipeline:
1. **Hardware & Core Inventory**: Queries physical/logical CPU cores, system RAM, GPU Streaming Multiprocessors (SMs), total CUDA cores, and dedicated VRAM.
2. **Deep Residual Architecture**: Defines a configurable Deep Residual MLP with LayerNorm, GELU activations, Cross-Entropy loss, and `AdamW` optimization.
3. **Stage 1 (CPU Baseline)**: Trains on `device='cpu'` while verifying `0 MB` GPU VRAM usage and measuring CPU thread saturation.
4. **Stage 2 (CUDA Full-Load Stress Test)**: Trains a ~222M-parameter model on `device='cuda:0'`, allocating **~4.5 GB of dedicated VRAM** and synchronizing per batch for 15 seconds while logging live CUDA core load (%), VRAM (MiB), power draw (W), and temperature (°C).
5. **Automated Comparison & Verdict**: Outputs a side-by-side comparison table, plain-English summary, and full VRAM garbage collection (`0 MB` leaks).

## 1. Hardware & Core Inventory Check

In [ ]:
import gc
import os
import sys
import time
import platform
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

def free_gpu_memory():
    """Clears any lingering test tensors and autograd graphs from GPU VRAM."""
    target_vars = [
        "cpu_model", "cpu_optimizer", "cpu_x", "cpu_y",
        "gpu_model", "gpu_optimizer", "dataset_cache_vram", "labels_cache_vram",
        "batch_x", "batch_y", "outputs", "loss", "_out", "_loss", "out"
    ]
    g = globals()
    for name in target_vars:
        if name in g:
            del g[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def get_nvidia_smi_stats():
    """Queries live GPU telemetry directly from the NVIDIA driver via nvidia-smi."""
    cmd = [
        "nvidia-smi",
        "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,temperature.gpu,clocks.current.sm",
        "--format=csv,noheader,nounits"
    ]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.strip()
        if not out:
            return None
        parts = [p.strip() for p in out.split(",")]
        return {
            "name": parts[0],
            "gpu_util": float(parts[1]) if parts[1] != "[N/A]" else 0.0,
            "mem_util": float(parts[2]) if parts[2] != "[N/A]" else 0.0,
            "vram_used_mib": float(parts[3]) if parts[3] != "[N/A]" else 0.0,
            "vram_total_mib": float(parts[4]) if parts[4] != "[N/A]" else 0.0,
            "power_w": float(parts[5]) if parts[5] != "[N/A]" else 0.0,
            "temp_c": float(parts[6]) if parts[6] != "[N/A]" else 0.0,
            "clock_mhz": float(parts[7]) if len(parts) > 7 and parts[7] != "[N/A]" else 0.0,
        }
    except Exception:
        return None

# Ensure a 100% clean VRAM state before taking baseline measurements
free_gpu_memory()

print("=" * 78)
print(" SYSTEM & HARDWARE INVENTORY")
print("=" * 78)
print(f"Python Version    : {sys.version.split()[0]} ({platform.system()} {platform.release()})")

logical_cores = os.cpu_count() or 1
phys_cores = psutil.cpu_count(logical=False) if HAS_PSUTIL else "N/A"
sys_ram_gb = round(psutil.virtual_memory().total / (1024**3), 2) if HAS_PSUTIL else "N/A"
print(f"CPU Cores         : {phys_cores} Physical Cores / {logical_cores} Logical Threads")
print(f"System RAM        : {sys_ram_gb} GB")
print("-" * 78)

print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    sm_count = props.multi_processor_count
    cores_per_sm = 128 if props.major >= 8 else 64
    total_cuda_cores = sm_count * cores_per_sm
    total_vram_mib = round(props.total_memory / (1024**2))
    total_vram_gb = round(props.total_memory / 1e9, 2)
    smi = get_nvidia_smi_stats()

    print(f"Dedicated GPU     : {props.name} (Compute Capability {props.major}.{props.minor})")
    print(f"GPU Core Count    : {sm_count} Streaming Multiprocessors (SMs) = {total_cuda_cores:,} CUDA Cores")
    print(f"Total Dedicated   : {total_vram_mib:,} MiB ({total_vram_gb} GB VRAM)")
    if smi:
        print(f"Clean Idle State  : {smi['gpu_util']:.0f}% GPU Load | {smi['vram_used_mib']:.0f} MiB Used | {smi['power_w']:.1f} W | {smi['temp_c']:.0f}°C")
else:
    print("WARNING: CUDA is not available in this kernel.")
print("=" * 78)

## 2. Define the Deep Residual Neural Network Architecture

In [ ]:
class DeepResidualTrainingNet(nn.Module):
    """Deep Residual MLP designed to benchmark training compute, gradients, and VRAM."""
    def __init__(self, dim=4096, num_blocks=6, num_classes=1000):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU()
        )
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, dim),
                nn.LayerNorm(dim),
                nn.GELU(),
                nn.Linear(dim, dim),
                nn.LayerNorm(dim)
            )
            for _ in range(num_blocks)
        ])
        self.act = nn.GELU()
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        h = self.input_layer(x)
        for block in self.blocks:
            h = self.act(h + block(h))
        return self.head(h)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("DeepResidualTrainingNet architecture defined.")

## 3. Stage 1: CPU-Only Training Baseline (`device='cpu'`)

In [ ]:
free_gpu_memory()

print("=" * 88)
print(" STAGE 1: RUNNING MODEL TRAINING ON CPU (device='cpu')")
print("=" * 88)

cpu_dim = 2048
cpu_batch_size = 256
cpu_steps = 6

cpu_model = DeepResidualTrainingNet(dim=cpu_dim, num_blocks=4, num_classes=500).to("cpu")
cpu_params_m = count_parameters(cpu_model) / 1e6
cpu_optimizer = optim.AdamW(cpu_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

cpu_x = torch.randn(cpu_batch_size, cpu_dim, device="cpu")
cpu_y = torch.randint(0, 500, (cpu_batch_size,), device="cpu")
print(f"CPU Model Initialized : {cpu_params_m:.1f}M trainable parameters | Batch Size: {cpu_batch_size}")
print("-" * 88)

if HAS_PSUTIL:
    psutil.cpu_percent(interval=None)

cpu_metrics = {
    "params_m": cpu_params_m,
    "cpu_percent_peak": 0.0,
    "gpu_util_peak": 0.0,
    "pytorch_vram_mb": 0.0,
    "power_w_peak": 0.0,
    "samples_per_sec": 0.0
}

t0 = time.time()
for step in range(1, cpu_steps + 1):
    step_t0 = time.time()
    cpu_optimizer.zero_grad(set_to_none=True)
    out = cpu_model(cpu_x)
    loss = criterion(out, cpu_y)
    loss.backward()
    cpu_optimizer.step()
    step_dt = time.time() - step_t0

    cpu_load = psutil.cpu_percent(interval=None) if HAS_PSUTIL else 0.0
    active_cpu_threads = round((cpu_load / 100.0) * logical_cores, 1) if HAS_PSUTIL else "N/A"
    smi = get_nvidia_smi_stats()
    gpu_util = smi["gpu_util"] if smi else 0.0
    gpu_power = smi["power_w"] if smi else 0.0
    pytorch_vram = round(torch.cuda.memory_allocated() / (1024**2), 1) if torch.cuda.is_available() else 0.0

    cpu_metrics["cpu_percent_peak"] = max(cpu_metrics["cpu_percent_peak"], cpu_load)
    cpu_metrics["gpu_util_peak"] = max(cpu_metrics["gpu_util_peak"], gpu_util)
    cpu_metrics["pytorch_vram_mb"] = max(cpu_metrics["pytorch_vram_mb"], pytorch_vram)
    cpu_metrics["power_w_peak"] = max(cpu_metrics["power_w_peak"], gpu_power)

    print(
        f"[Step {step}/{cpu_steps}] Time: {step_dt*1000:5.0f} ms | "
        f"CPU Threads: ~{active_cpu_threads}/{logical_cores} ({cpu_load:3.0f}%) | "
        f"GPU Load: {gpu_util:2.0f}% (0 Training Cores) | "
        f"PyTorch VRAM: {pytorch_vram:.0f} MB"
    )

cpu_total_time = time.time() - t0
cpu_metrics["samples_per_sec"] = (cpu_steps * cpu_batch_size) / cpu_total_time

print("-" * 88)
print(f"CPU Training Complete in {cpu_total_time:.1f}s | Throughput: {cpu_metrics['samples_per_sec']:.1f} samples/sec")
print(f"Verified: PyTorch GPU Training VRAM stayed at {cpu_metrics['pytorch_vram_mb']:.0f} MB during CPU training.")
print("=" * 88)

free_gpu_memory()

## 4. Stage 2: Full-Load GPU Training Stress Test (`device='cuda:0'`)

In [ ]:
assert torch.cuda.is_available(), "CUDA is not available! Check your active kernel."
free_gpu_memory()

print("=" * 96)
print(" STAGE 2: FULL-LOAD TRAINING ON NVIDIA CUDA GPU (device='cuda:0')")
print("=" * 96)

gpu_dim = 4096
gpu_blocks = 6
gpu_batch_size = 1024

print("Allocating Deep Residual Network and training batches into Dedicated GPU VRAM...")
gpu_model = DeepResidualTrainingNet(dim=gpu_dim, num_blocks=gpu_blocks, num_classes=1000).to("cuda")
gpu_params_m = count_parameters(gpu_model) / 1e6
gpu_optimizer = optim.AdamW(gpu_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

dataset_cache_vram = torch.randn(16, gpu_batch_size, gpu_dim, device="cuda")
labels_cache_vram = torch.randint(0, 1000, (16, gpu_batch_size), device="cuda")

# Warm-up 1 step so AdamW state buffers are initialized in VRAM before timing starts
gpu_optimizer.zero_grad(set_to_none=True)
_out = gpu_model(dataset_cache_vram[0])
_loss = criterion(_out, labels_cache_vram[0])
_loss.backward()
gpu_optimizer.step()
del _out, _loss
torch.cuda.synchronize()

initial_vram_mb = torch.cuda.memory_allocated() / (1024**2)
reserved_vram_mb = torch.cuda.memory_reserved() / (1024**2)
print(f"GPU Model Initialized : {gpu_params_m:.1f}M trainable parameters | Batch Size: {gpu_batch_size}")
print(f"Active Training VRAM  : {initial_vram_mb:,.0f} MB Allocated (Reserved: {reserved_vram_mb:,.0f} MB / {total_vram_mib:,} MiB)")
print("-" * 96)

gpu_metrics = {
    "params_m": gpu_params_m,
    "gpu_util_samples": [],
    "gpu_util_peak": 0.0,
    "active_cores_peak": 0,
    "pytorch_vram_peak_mb": 0.0,
    "driver_vram_peak_mib": 0.0,
    "power_w_peak": 0.0,
    "temp_c_peak": 0.0,
    "samples_per_sec": 0.0,
    "steps_completed": 0
}

DURATION_SEC = 15.0
t_start = time.time()
last_print = t_start
step = 0

while (time.time() - t_start) < DURATION_SEC:
    idx = step % 16
    batch_x = dataset_cache_vram[idx]
    batch_y = labels_cache_vram[idx]

    gpu_optimizer.zero_grad(set_to_none=True)
    outputs = gpu_model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()
    gpu_optimizer.step()

    # Synchronize per batch so timing is exact and the CUDA queue never overfills
    torch.cuda.synchronize()
    step += 1

    now = time.time()
    if (now - last_print) >= 2.0 or (now - t_start) >= DURATION_SEC:
        smi = get_nvidia_smi_stats()
        gpu_util = smi["gpu_util"] if smi else 0.0
        driver_vram = smi["vram_used_mib"] if smi else 0.0
        power_w = smi["power_w"] if smi else 0.0
        temp_c = smi["temp_c"] if smi else 0.0

        pt_peak_mb = torch.cuda.max_memory_allocated() / (1024**2)
        est_active_cores = int(round((gpu_util / 100.0) * total_cuda_cores))

        gpu_metrics["gpu_util_samples"].append(gpu_util)
        gpu_metrics["gpu_util_peak"] = max(gpu_metrics["gpu_util_peak"], gpu_util)
        gpu_metrics["active_cores_peak"] = max(gpu_metrics["active_cores_peak"], est_active_cores)
        gpu_metrics["pytorch_vram_peak_mb"] = max(gpu_metrics["pytorch_vram_peak_mb"], pt_peak_mb)
        gpu_metrics["driver_vram_peak_mib"] = max(gpu_metrics["driver_vram_peak_mib"], driver_vram)
        gpu_metrics["power_w_peak"] = max(gpu_metrics["power_w_peak"], power_w)
        gpu_metrics["temp_c_peak"] = max(gpu_metrics["temp_c_peak"], temp_c)

        print(
            f"[t={now - t_start:4.1f}s | Step {step:3d}] "
            f"GPU Cores: {gpu_util:3.0f}% (~{est_active_cores:4d}/{total_cuda_cores} CUDA cores) | "
            f"VRAM Used: {driver_vram:4.0f} MiB (PyTorch Peak: {pt_peak_mb:4.0f} MB) | "
            f"Power: {power_w:4.1f}W | Temp: {temp_c:2.0f}°C"
        )
        last_print = now

gpu_total_time = time.time() - t_start
gpu_metrics["steps_completed"] = step
gpu_metrics["samples_per_sec"] = (step * gpu_batch_size) / gpu_total_time

print("-" * 96)
print(f"GPU Stress Test Completed Cleanly in {gpu_total_time:.1f}s ({step} full forward+backward steps)!")
print("=" * 96)

## 5. Final Comparison Table, Plain-English Summary & VRAM Cleanup

In [ ]:
speedup = gpu_metrics["samples_per_sec"] / max(cpu_metrics["samples_per_sec"], 1e-6)
param_ratio = gpu_metrics["params_m"] / max(cpu_metrics["params_m"], 1e-6)
effective_compute_speedup = speedup * param_ratio

c1_hw = f"{logical_cores} CPU Threads"
c2_hw = f"{sm_count} SMs ({total_cuda_cores:,} CUDA Cores)"
c1_mod = f"{cpu_metrics['params_m']:.1f}M Parameters"
c2_mod = f"{gpu_metrics['params_m']:.1f}M Parameters ({param_ratio:.1f}x)"
c1_load = f"{cpu_metrics['gpu_util_peak']:.0f}% (0 Training Cores)"
c2_load = f"{gpu_metrics['gpu_util_peak']:.0f}% (~{gpu_metrics['active_cores_peak']:,} Cores)"
c1_vram = f"{cpu_metrics['pytorch_vram_mb']:.0f} MB"
c2_vram = f"{gpu_metrics['pytorch_vram_peak_mb']:,.0f} MB ({gpu_metrics['pytorch_vram_peak_mb']/1024:.2f} GB)"
c1_tot = "Idle / Display Only"
c2_tot = f"{gpu_metrics['driver_vram_peak_mib']:,.0f} / {total_vram_mib:,} MiB"
c1_pwr = f"{cpu_metrics['power_w_peak']:.1f} W"
c2_pwr = f"{gpu_metrics['power_w_peak']:.1f} W"
c1_spd = f"{cpu_metrics['samples_per_sec']:,.1f} samples/s"
c2_spd = f"{gpu_metrics['samples_per_sec']:,.1f} samples/s ({speedup:.1f}x)"

print("=" * 90)
print(" DETAILED COMPARISON: CPU TRAINING vs. NVIDIA RTX 4050 GPU TRAINING")
print("=" * 90)
print(f"{'Metric':<34} | {'Stage 1: CPU Only':<23} | {'Stage 2: RTX 4050 GPU':<26}")
print("-" * 90)
print(f"{'Compute Hardware Used':<34} | {c1_hw:<23} | {c2_hw:<26}")
print(f"{'Neural Network Size':<34} | {c1_mod:<23} | {c2_mod:<26}")
print(f"{'Peak GPU CUDA Core Load (%)':<34} | {c1_load:<23} | {c2_load:<26}")
print(f"{'PyTorch Peak Training VRAM':<34} | {c1_vram:<23} | {c2_vram:<26}")
print(f"{'Total Dedicated GPU Memory Used':<34} | {c1_tot:<23} | {c2_tot:<26}")
print(f"{'Peak GPU Power Draw':<34} | {c1_pwr:<23} | {c2_pwr:<26}")
print(f"{'Training Throughput (Speed)':<34} | {c1_spd:<23} | {c2_spd:<26}")
print("=" * 90)

gpu_working = (
    gpu_metrics["pytorch_vram_peak_mb"] > 2000
    and gpu_metrics["gpu_util_peak"] >= 50
    and speedup >= 2.0
)
status_text = "YES — 100% WORKING AT FULL POWER" if gpu_working else "NO — CHECK GPU SETTINGS"
active_cpu_est = round((cpu_metrics["cpu_percent_peak"] / 100.0) * logical_cores, 1)

print("\n" + "#" * 78)
print(" SIMPLE RESULTS SUMMARY (EASY TO READ)")
print("#" * 78)
print(f" 1. Is RTX 4050 Training Models? : {status_text}")
print(f" 2. Cores Used (GPU vs CPU)      : GPU used {gpu_metrics['active_cores_peak']:,} / {total_cuda_cores:,} CUDA cores ({gpu_metrics['gpu_util_peak']:.0f}% load)")
print(f"                                   CPU test used ~{active_cpu_est} / {logical_cores} CPU threads (0 CUDA cores)")
print(f" 3. Memory Used (GPU vs CPU)     : GPU training allocated {gpu_metrics['pytorch_vram_peak_mb']:,.0f} MB ({gpu_metrics['pytorch_vram_peak_mb']/1024:.2f} GB) of 6 GB VRAM")
print(f"                                   CPU training allocated {cpu_metrics['pytorch_vram_mb']:.0f} MB of GPU VRAM (ran in system RAM)")
print(f" 4. Why It Is NOT Integrated GPU : Integrated GPU only has ~512 MB; this test used {gpu_metrics['driver_vram_peak_mib']:,.0f} MiB")
print(f" 5. Speed & Power Difference     : RTX 4050 pulled {gpu_metrics['power_w_peak']:.1f}W and ran {speedup:.1f}x faster in samples/sec")
print(f"                                   (~{effective_compute_speedup:.0f}x faster total neural network math on a {param_ratio:.1f}x larger model)")
print("#" * 78)

# Complete VRAM and Autograd Graph Cleanup
free_gpu_memory()
print(f"\n[Cleanup Done] Freed all test tensors & autograd graphs. Current PyTorch VRAM in use: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")